<h2 style="color:cyan" align="center">LLM Project in Retail Industry

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

GOOGLE_API_KEY = ""

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    temperature=0.3,  # Gemini 3.0+ defaults to 1.0
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_classic.chains import create_sql_query_chain

In [ ]:
db_user = "root"
db_password = "mysqlroot"
db_host = "localhost"
db_name = "atliq_tshirts"

db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}", sample_rows_in_table_info=3)

In [ ]:
print(db.table_info)


CREATE TABLE discounts (
	discount_id INTEGER NOT NULL AUTO_INCREMENT, 
	t_shirt_id INTEGER NOT NULL, 
	pct_discount DECIMAL(5, 2), 
	PRIMARY KEY (discount_id), 
	CONSTRAINT discounts_ibfk_1 FOREIGN KEY(t_shirt_id) REFERENCES t_shirts (t_shirt_id), 
	CONSTRAINT discounts_chk_1 CHECK ((`pct_discount` between 0 and 100))
)ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4

/*
3 rows from discounts table:
discount_id	t_shirt_id	pct_discount
1	1	10.00
2	2	15.00
3	3	20.00
*/


CREATE TABLE t_shirts (
	t_shirt_id INTEGER NOT NULL AUTO_INCREMENT, 
	brand ENUM('Van Huesen','Levi','Nike','Adidas') NOT NULL, 
	color ENUM('Red','Blue','Black','White') NOT NULL, 
	size ENUM('XS','S','M','L','XL') NOT NULL, 
	price INTEGER, 
	stock_quantity INTEGER NOT NULL, 
	PRIMARY KEY (t_shirt_id), 
	CONSTRAINT t_shirts_chk_1 CHECK ((`price` between 10 and 50))
)ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4

/*
3 rows from t_shirts table:
t_shirt_id	brand	color	size	price	stock

In [ ]:
query_chain = create_sql_query_chain(llm, db)
sql_query = query_chain.invoke({"question": "How many t-shirts do we have left for Nike in extra small size and white colour?"})
print(sql_query) 

Question: How many t-shirts do we have left for Nike in extra small size and white colour?
SQLQuery: SELECT `stock_quantity` FROM `t_shirts` WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'White' LIMIT 1;


In [ ]:
def clean_sql_query(raw_query):
    # Delete "SQLQuery:" 
    if "SQLQuery:" in raw_query:
        raw_query = raw_query.split("SQLQuery:")[-1]
    clean_query = raw_query.replace("```sql", "").replace("```", "").strip()

    return clean_query

In [ ]:
sql_query = clean_sql_query(sql_query)
result = db.run(sql_query)
print(sql_query)
print(f"Result is: {result}")

SELECT `stock_quantity` FROM `t_shirts` WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'White' LIMIT 1;
Result is: [(88,)]


In [ ]:
sql_query_2 = query_chain.invoke({"question": "How much is the inventory cost for all small-sized t-shirts?"})

In [ ]:
print(sql_query_2) 

Question: How much is the inventory cost for all small-sized t-shirts?
SQLQuery: SELECT SUM(`price` * `stock_quantity`) FROM `t_shirts` WHERE `size` = 'S'


In [ ]:
sql_query_2 = clean_sql_query(sql_query_2)
result_2 = db.run(sql_query_2)
print(sql_query_2)
print(f"Result is: {result_2}")

SELECT SUM(`price` * `stock_quantity`) FROM `t_shirts` WHERE `size` = 'S'
Result is: [(Decimal('19919'),)]


In [ ]:
sql_query_3 = query_chain.invoke({"question": "How much revenue will our store generate (after discounts) if we sell all the Levi's T-shirts today?"})

In [ ]:
print(sql_query_3) 

Question: How much revenue will our store generate (after discounts) if we sell all the Levi's T-shirts today?
SQLQuery: SELECT SUM(t.`price` * t.`stock_quantity` * (1 - COALESCE(d.`pct_discount`, 0) / 100)) AS `total_revenue` FROM `t_shirts` t LEFT JOIN `discounts` d ON t.`t_shirt_id` = d.`t_shirt_id` WHERE t.`brand` = 'Levi'


In [ ]:
sql_query_3 = clean_sql_query(sql_query_3)
result_3 = db.run(sql_query_3)
print(sql_query_3)
print(f"Result is: {result_3}")

SELECT SUM(t.`price` * t.`stock_quantity` * (1 - COALESCE(d.`pct_discount`, 0) / 100)) AS `total_revenue` FROM `t_shirts` t LEFT JOIN `discounts` d ON t.`t_shirt_id` = d.`t_shirt_id` WHERE t.`brand` = 'Levi'
Result is: [(Decimal('22618.600000'),)]


In [ ]:
sql_query_4 = query_chain.invoke({"question": "SELECT SUM(price * stock_quantity) FROM t_shirts WHERE brand = 'Levi'"})

In [ ]:
print(sql_query_4) 

Question: SELECT SUM(price * stock_quantity) FROM t_shirts WHERE brand = 'Levi'
SQLQuery: SELECT SUM(`price` * `stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Levi'


In [ ]:
sql_query_4 = clean_sql_query(sql_query_4)
result_4 = db.run(sql_query_4)
print(sql_query_4)
print(f"Result is: {result_4}")

SELECT SUM(`price` * `stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Levi'
Result is: [(Decimal('23029'),)]


In [ ]:
sql_query_5 = query_chain.invoke({"question": "How many white Levi's t-shirts do we have available?"})

In [ ]:
sql_query_5 = clean_sql_query(sql_query_5)
result_5 = db.run(sql_query_5)
print(sql_query_5)
print(f"Result is: {result_5}")

SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Levi' AND `color` = 'White'
Result is: [(Decimal('281'),)]


In [ ]:
sql_query_6 = query_chain.invoke({"question": "What is the percentage contribution of Nike to the total stock quantity of the entire inventory?"})

In [ ]:
sql_query_6 = clean_sql_query(sql_query_6)
result_6 = db.run(sql_query_6)
print(sql_query_6)
print(f"Result is: {result_6}")

SELECT (SUM(CASE WHEN `brand` = 'Nike' THEN `stock_quantity` ELSE 0 END) * 100.0 / SUM(`stock_quantity`)) AS `percentage` FROM `t_shirts`
Result is: [(Decimal('30.52995'),)]


In [ ]:
sql_query_7 = query_chain.invoke({"question": "List all brands that have more than 50 total items in stock, \
                                  but only considering White and Blue colours"})

In [ ]:
sql_query_7 = clean_sql_query(sql_query_7)
result_7 = db.run(sql_query_7)
print(sql_query_7)
print(f"Result is: {result_7}")

SELECT `brand`, SUM(`stock_quantity`) AS `total_stock` FROM `t_shirts` WHERE `color` IN ('White', 'Blue') GROUP BY `brand` HAVING `total_stock` > 50
Result is: [('Van Huesen', Decimal('363')), ('Levi', Decimal('347')), ('Nike', Decimal('711')), ('Adidas', Decimal('486'))]


In [ ]:
sql_query_8 = query_chain.invoke({"question": "List all brands that have more than 500 total items in stock, \
                                  but only considering White and Blue colours"})

In [ ]:
sql_query_8 = clean_sql_query(sql_query_8)
result_8 = db.run(sql_query_8)
print(sql_query_8)
print(f"Result is: {result_8}")

SELECT `brand` FROM `t_shirts` WHERE `color` IN ('White', 'Blue') GROUP BY `brand` HAVING SUM(`stock_quantity`) > 500
Result is: [('Nike',)]


In [ ]:
sufficient_sql_query_8 = query_chain.invoke({"question": "SELECT `brand`, SUM(`stock_quantity`) FROM atliq_tshirts.t_shirts WHERE `color` \
                                             IN ('White', 'Blue') GROUP BY `brand` HAVING SUM(`stock_quantity`) > 500;"})

In [ ]:
sufficient_sql_query_8 = clean_sql_query(sufficient_sql_query_8)
sufficient_result_8 = db.run(sufficient_sql_query_8)
print(sufficient_sql_query_8)
print(f"Result is: {sufficient_result_8}")

SELECT `brand`, SUM(`stock_quantity`) FROM `t_shirts` WHERE `color` IN ('White', 'Blue') GROUP BY `brand` HAVING SUM(`stock_quantity`) > 500 LIMIT 5;
Result is: [('Nike', Decimal('711'))]


In [ ]:
sql_query_9 = query_chain.invoke({"question": "Find all colours that are not available in the Adidas brand."})

In [ ]:
sql_query_9 = clean_sql_query(sql_query_9)
result_9 = db.run(sql_query_9)
print(sql_query_9)
print(f"Result is: {result_9}")

SELECT DISTINCT `color` FROM `t_shirts` WHERE `color` NOT IN (SELECT `color` FROM `t_shirts` WHERE `brand` = 'Adidas')
Result is: 


In [ ]:
sql_query_10 = query_chain.invoke({"question": "Which brand has the highest potential revenue (price multiplied by stock) \
                                   for Small (S) size t-shirts?"})

In [ ]:
sql_query_10 = clean_sql_query(sql_query_10)
result_10 = db.run(sql_query_10)
print(sql_query_10)
print(f"Result is: {result_10}")

SELECT `brand`, (`price` * `stock_quantity`) AS `potential_revenue` FROM `t_shirts` WHERE `size` = 'S' ORDER BY `potential_revenue` DESC LIMIT 1;
Result is: [('Levi', 3290)]


In [ ]:
sql_query_10_correct = query_chain.invoke({"question": "SELECT brand, SUM(price * stock_quantity) as potential_revenue \
                                           FROM atliq_tshirts.t_shirts WHERE size = 'S' GROUP BY brand \
                                           ORDER BY potential_revenue DESC LIMIT 1;"})

In [ ]:
sql_query_10_correct = clean_sql_query(sql_query_10_correct)
result_10_correct = db.run(sql_query_10_correct)
print(sql_query_10_correct)
print(f"Result is: {result_10_correct}")

SELECT `brand`, SUM(`price` * `stock_quantity`) AS `potential_revenue` FROM `t_shirts` WHERE `size` = 'S' GROUP BY `brand` ORDER BY `potential_revenue` DESC LIMIT 1;
Result is: [('Adidas', Decimal('9057'))]


In [ ]:
sql_query_11 = query_chain.invoke({"question": "Compare the average price of Nike t-shirts versus Adidas t-shirts. \
                                   Which one is more expensive on average?"})

In [ ]:
sql_query_11 = clean_sql_query(sql_query_11)
result_11 = db.run(sql_query_11)
print(sql_query_11)
print(f"Result is: {result_11}")

SELECT `brand`, AVG(`price`) AS `avg_price` FROM `t_shirts` WHERE `brand` IN ('Nike', 'Adidas') GROUP BY `brand`
Result is: [('Nike', Decimal('28.4375')), ('Adidas', Decimal('28.9375'))]


In [ ]:
sql_query_12 = query_chain.invoke({"question": "Tìm áo cỡ đại"})

In [ ]:
sql_query_12 = clean_sql_query(sql_query_12)
result_12 = db.run(sql_query_12)
print(sql_query_12)
print(f"Result is: {result_12}")

SELECT `t_shirt_id`, `brand`, `color`, `size`, `price` FROM `t_shirts` WHERE `size` = 'XL' LIMIT 5
Result is: [(3, 'Adidas', 'Blue', 'XL', 22), (4, 'Adidas', 'Black', 'XL', 31), (6, 'Levi', 'Red', 'XL', 20), (7, 'Adidas', 'Red', 'XL', 13), (9, 'Levi', 'Black', 'XL', 31)]


In [ ]:
sql_query_12_correct = query_chain.invoke({"question": "SELECT * FROM t_shirts WHERE size = 'XL' và không giới hạn số lượng trả về"})

In [ ]:
sql_query_12_correct = clean_sql_query(sql_query_12_correct)
result_12_correct = db.run(sql_query_12_correct)
print(sql_query_12_correct)
print(f"Result is: {result_12_correct}")

SELECT `brand`, `color`, `size`, `price`, `stock_quantity` FROM `t_shirts` WHERE `size` = 'XL'
Result is: [('Adidas', 'Blue', 'XL', 22, 98), ('Adidas', 'Black', 'XL', 31, 76), ('Levi', 'Red', 'XL', 20, 75), ('Adidas', 'Red', 'XL', 13, 83), ('Levi', 'Black', 'XL', 31, 21), ('Van Huesen', 'White', 'XL', 31, 64), ('Nike', 'Black', 'XL', 19, 59), ('Van Huesen', 'Black', 'XL', 31, 100), ('Nike', 'Blue', 'XL', 21, 76), ('Nike', 'Red', 'XL', 16, 88), ('Nike', 'White', 'XL', 39, 87), ('Van Huesen', 'Red', 'XL', 32, 96), ('Van Huesen', 'Blue', 'XL', 27, 38)]


In [ ]:
sql_query_13 = query_chain.invoke({"question": "Tổng giá trị kho của Nike"})

In [ ]:
sql_query_13 = clean_sql_query(sql_query_13)
result_13 = db.run(sql_query_13)
print(sql_query_13)
print(f"Result is: {result_13}")

SELECT SUM(`price` * `stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Nike'
Result is: [(Decimal('29438'),)]


In [ ]:
sql_query_14 = query_chain.invoke({"question": "Áo màu đen"})

In [ ]:
sql_query_14 = clean_sql_query(sql_query_14)
result_14 = db.run(sql_query_14)
print(sql_query_14)
print(f"Result is: {result_14}")

SELECT `brand`, `size`, `price`, `stock_quantity` FROM `t_shirts` WHERE `color` = 'Black' LIMIT 5
Result is: [('Adidas', 'XL', 31, 76), ('Adidas', 'S', 19, 100), ('Levi', 'XL', 31, 21), ('Nike', 'XL', 19, 59), ('Van Huesen', 'XL', 31, 100)]


In [ ]:
sql_query_15 = query_chain.invoke({"question": "Liệt kê tất cả áo màu đen, không giới hạn số lượng dòng trả về"})

In [ ]:
sql_query_15 = clean_sql_query(sql_query_15)
result_15 = db.run(sql_query_15)
print(sql_query_15)
print(f"Result is: {result_15}")

SELECT `t_shirt_id`, `brand`, `color`, `size`, `price`, `stock_quantity` FROM `t_shirts` WHERE `color` = 'Black'
Result is: [(4, 'Adidas', 'Black', 'XL', 31, 76), (8, 'Adidas', 'Black', 'S', 19, 100), (9, 'Levi', 'Black', 'XL', 31, 21), (13, 'Nike', 'Black', 'XL', 19, 59), (14, 'Van Huesen', 'Black', 'XL', 31, 100), (19, 'Nike', 'Black', 'L', 50, 27), (31, 'Nike', 'Black', 'XS', 23, 13), (35, 'Adidas', 'Black', 'L', 39, 30), (37, 'Van Huesen', 'Black', 'L', 49, 69), (42, 'Levi', 'Black', 'L', 22, 16), (59, 'Nike', 'Black', 'M', 10, 30), (73, 'Van Huesen', 'Black', 'XS', 25, 61), (75, 'Van Huesen', 'Black', 'M', 29, 12), (97, 'Levi', 'Black', 'M', 46, 96), (98, 'Van Huesen', 'Black', 'S', 50, 35)]


## Few Shot Learning to fix the answer 

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2') 

d:\AI_projects\llm-project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3650.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
few_shots = [
    {
        'Question' : "How much is the inventory cost for all small-sized t-shirts?",
        'SQLQuery' : "SELECT SUM(`price` * `stock_quantity`) FROM `t_shirts` WHERE `size` = 'S'",
        'SQLResult': "Result of the SQL query",
        'Answer' : result_2
    },
    {
        'Question' : "How many white Levi's t-shirts do we have available?",
        'SQLQuery' : "SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Levi' AND `color` = 'White'",
        'SQLResult': "Result of the SQL query",
        'Answer' : result_5
    },
    {
        'Question' : "List all brands that have more than 500 total items in stock, but only considering White and Blue colours",
        'SQLQuery' : """SELECT `brand`, SUM(`stock_quantity`) FROM `t_shirts` WHERE `color` IN ('White', 'Blue') 
                        GROUP BY `brand` HAVING SUM(`stock_quantity`) > 500""",
        'SQLResult': "Result of the SQL query",
        'Answer' : sufficient_result_8
    },
    {
        'Question' : "Find all colours that are not available in the Adidas brand.",
        'SQLQuery' : """SELECT DISTINCT `color` FROM `t_shirts` WHERE `color` 
                        NOT IN (SELECT `color` FROM `t_shirts` WHERE `brand` = 'Adidas')""",
        'SQLResult': "Result of the SQL query",
        'Answer' : "All available colours in the inventory are also offered by Adidas, so there are no colours exclusive to other brands."
    },
    {
        'Question' : "Which brand has the highest potential revenue (price multiplied by stock) for Small (S) size t-shirts?",
        'SQLQuery' : """SELECT `brand`, SUM(`price` * `stock_quantity`) AS `potential_revenue` FROM `t_shirts` 
                        WHERE `size` = 'S' GROUP BY `brand` ORDER BY `potential_revenue` DESC LIMIT 1""",
        'SQLResult': "Result of the SQL query",
        'Answer' : result_10_correct
    },
    {
        'Question' : "Tìm áo cỡ đại",
        'SQLQuery' : "SELECT `brand`, `color`, `size`, `price`, `stock_quantity` FROM `t_shirts` WHERE `size` = 'XL'",
        'SQLResult': "Result of the SQL query",
        'Answer' : result_12_correct
    }
]

In [ ]:
to_vectorise = [" ".join(example.values()) for example in few_shots]
to_vectorise

["How much is the inventory cost for all small-sized t-shirts? SELECT SUM(`price` * `stock_quantity`) FROM `t_shirts` WHERE `size` = 'S' Result of the SQL query [(Decimal('19919'),)]",
 "How many white Levi's t-shirts do we have available? SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Levi' AND `color` = 'White' Result of the SQL query [(Decimal('281'),)]",
 "List all brands that have more than 500 total items in stock, but only considering White and Blue colors SELECT `brand`, SUM(`stock_quantity`) FROM `t_shirts` WHERE `color` IN ('White', 'Blue') \n                        GROUP BY `brand` HAVING SUM(`stock_quantity`) > 500 Result of the SQL query [('Nike', Decimal('711'))]",
 "Find all colours that are not available in the Adidas brand. SELECT DISTINCT `color` FROM `t_shirts` WHERE `color` \n                        NOT IN (SELECT `color` FROM `t_shirts` WHERE `brand` = 'Adidas') Result of the SQL query All available colors in the inventory are also offered by Adidas,

In [ ]:
from langchain_chroma import Chroma
vectorstore = Chroma.from_texts(to_vectorise, embedding=embeddings, metadatas=few_shots)

In [ ]:
from langchain_core.example_selectors.semantic_similarity import SemanticSimilarityExampleSelector

example_selector = SemanticSimilarityExampleSelector(
    vectorstore=vectorstore,
    k=2,
)

In [ ]:
example_selector.select_examples({"Question": "How many Adidas T shirts I have left in my store?"})

[{'SQLResult': 'Result of the SQL query',
  'Question': "How many white Levi's t-shirts do we have available?",
  'SQLQuery': "SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Levi' AND `color` = 'White'",
  'Answer': "[(Decimal('281'),)]"},
 {'Question': "How many white Levi's t-shirts do we have available?",
  'SQLResult': 'Result of the SQL query',
  'SQLQuery': "SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Levi' AND `color` = 'White'",
  'Answer': "[(Decimal('281'),)]"}]

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# Prefix - MySQL instruction prompt
mysql_prompt = """You are a MySQL expert. Given an input question, create a syntactically correct MySQL query to answer the user's question.
Unless the user specifies a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause. 
Use the following table schema:
{table_info}
"""

# Example Prompt 
example_prompt = PromptTemplate(
    input_variables=["Question", "SQLQuery", "SQLResult", "Answer"],
    template="\nQuestion: {Question}\nSQLQuery: {SQLQuery}\nSQLResult: {SQLResult}\nAnswer: {Answer}"
)

# Suffix - Where to fill in the real questions
my_suffix = """
Question: {input}
SQLQuery: """

few_shot_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix=mysql_prompt,
    suffix=my_suffix,
    input_variables=["input", "table_info", "top_k"] 
)

In [ ]:
chain = create_sql_query_chain(llm, db, prompt=few_shot_prompt)

# Trial run
question = "Tìm áo size lớn nhất còn hàng"
response = chain.invoke({"question": question}) 

print(response)

SELECT `brand`, `color`, `size`, `price`, `stock_quantity` 
FROM `t_shirts` 
WHERE `size` = 'XL' AND `stock_quantity` > 0 
LIMIT 5;


In [ ]:
# Trial run
question_1 = "How many white Nike's t-shirts do we have available?"
response_1 = chain.invoke({"question": question_1}) 

print(response_1)

SELECT SUM(stock_quantity) FROM t_shirts WHERE brand = 'Nike' AND color = 'White';


In [ ]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

execute_query = QuerySQLDataBaseTool(db=db) # Create SQL execution tools
sql_query = chain.invoke({"question": "Tìm áo size lớn nhất còn hàng"})
result = execute_query.invoke(sql_query) # Execute that statement to retrieve data from MySQL.

print(result) 

C:\Users\tanld\AppData\Local\Temp\ipykernel_20728\3779942261.py:4: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the `langchain-community package and should be used instead. To use it run `pip install -U `langchain-community` and import as `from `langchain_community.tools import QuerySQLDatabaseTool``.
  execute_query = QuerySQLDataBaseTool(db=db)


[('Adidas', 'Blue', 'XL', 22, 98), ('Adidas', 'Black', 'XL', 31, 76), ('Levi', 'Red', 'XL', 20, 75), ('Adidas', 'Red', 'XL', 13, 83), ('Levi', 'Black', 'XL', 31, 21)]


In [ ]:
full_chain = chain | execute_query
data = full_chain.invoke({"question": "Tìm áo size lớn nhất còn hàng"})
print(data)

[('Adidas', 'Blue', 'XL', 22, 98), ('Adidas', 'Black', 'XL', 31, 76), ('Levi', 'Red', 'XL', 20, 75), ('Adidas', 'Red', 'XL', 13, 83), ('Levi', 'Black', 'XL', 31, 21)]


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# LLM read the result from DB to answer to human
answer_prompt = PromptTemplate.from_template(
    """Dựa vào câu hỏi của người dùng, câu lệnh SQL và kết quả trả về từ database dưới đây, hãy trả lời câu hỏi của người dùng một cách lịch sự.
        
    LƯU Ý QUAN TRỌNG: Trả lời bằng ĐÚNG NGÔN NGỮ mà người dùng đã hỏi.
    - Nếu khách hỏi tiếng Việt, hãy viết câu trả lời tự nhiên bằng tiếng Việt.
    - If the user asks in English, please respond in English.

    Câu hỏi: {question}
    SQL Query: {query}
    SQL Result: {result}
    Answer: """
)

# WORKFLOW ASSEMBLY:
# 1. Create SQL -> 2. Run SQL to retrieve results -> 3. Give results to AI to respond by English or Vietnamese
full_chain = (
    RunnablePassthrough.assign(query=chain) 
    | RunnablePassthrough.assign(result=lambda x: execute_query.invoke(x["query"]))
    | answer_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# Trial run
final_answer = full_chain.invoke({"question": "Tìm áo size lớn nhất còn hàng"})
print(final_answer)

Dưới đây là danh sách các mẫu áo size lớn nhất (XL) hiện đang còn hàng trong kho của chúng tôi:

1. Áo Adidas màu xanh dương, giá 22$, số lượng còn 98 chiếc.
2. Áo Adidas màu đen, giá 31$, số lượng còn 76 chiếc.
3. Áo Levi màu đỏ, giá 20$, số lượng còn 75 chiếc.
4. Áo Adidas màu đỏ, giá 13$, số lượng còn 83 chiếc.
5. Áo Levi màu đen, giá 31$, số lượng còn 21 chiếc.

Hy vọng bạn sẽ chọn được sản phẩm ưng ý!


In [ ]:
final_answer_1 = full_chain.invoke({"question": "How many white Nike's t-shirts do we have available?"})
print(final_answer_1)

We currently have 308 white Nike t-shirts available in stock.
